# <b style='font-size:30px;font-family:Arial'>Create a File-Embedding-Based Collection using Parquet Files</b>

File-embedding-based vector collections are designed for scenarios where you already have pre-computed embeddings in your Parquet files. The workflow is:
1. Configure the Ingestor pipeline with `.files()` and `.upsert()`
2. Execute the pipeline with `.run()` to create collection and ingest embeddings
3. Wait for collection indexing to complete
4. Perform similarity searches

**Key Learning:** For FILE-EMBEDDING-BASED collections, use `embedding_columns` parameter in `ExtractionSchema` and the method chain `.files().upsert().run()` where embeddings are already present in the files.

## <b style='font-size:20px;font-family:Arial'>1. Import Required Libraries</b>

In [ ]:
# Import Required Libraries
import sys
import os
import json
import glob
import getpass
from datetime import datetime
import pyarrow.parquet as pq

# TeradataML imports
from teradataml import create_context, remove_context, DataFrame

# Teradata Vector Store V2 API - Ingestor workflow
from teradatagenai import Collection, CollectionManager, ColumnInfo
from teradatagenai import BasicIngestor, ExtractionSchema, TeradataAI
from teradatagenai import LocalConfig, S3Config, AzureBlobConfig
from teradatagenai.vector_store.Ingestor import Ingestor
from teradatagenai.common.constants import CollectionType
from teradatasqlalchemy.types import VARCHAR, CLOB

print("✅ All imports successful!")
print("✓ Using Vector Store V2 API - Ingestor stepwise pipeline")
print("✓ Collection Type: FILE-EMBEDDING-BASED")

✅ All imports successful!
✓ Using Vector Store V2 API - Ingestor stepwise pipeline
✓ Collection Type: FILE-EMBEDDING-BASED


## <b style='font-size:20px;font-family:Arial'>2. Connect to Teradata Vantage</b>

In [ ]:
# Configure connection parameters
os.environ['TD_HOST'] = getpass.getpass('Enter Teradata Host: ')
os.environ['TD_USERNAME'] = getpass.getpass('Enter Teradata Username: ')
os.environ['TD_PASSWORD'] = getpass.getpass('Enter Teradata Password: ')
os.environ['TD_BASE_URL'] = getpass.getpass('Enter Base URL: ')
os.environ['TD_AUTH_MECH'] = 'BASIC'

# Create connection context
con = create_context()
print(f"✓ Connected to Teradata Vantage at {os.environ['TD_HOST']}")

# Verify service health
try:
    health = CollectionManager.health()
    print("✓ Vector Store service is healthy")
except Exception as e:
    print(f"Service status: {e}")

Authentication token is generated and set for the session.
✓ Connected to Teradata Vantage at 10.27.178.11
✓ Vector Store service is healthy


## <b style='font-size:20px;font-family:Arial'>3. Configure Embedding Model</b>

Specify the embedding model that will generate vector embeddings from the text content.

In [ ]:
# Configure embedding model (runtime generation)
os.environ["NVIDIA_API_KEY"] = getpass.getpass('Enter NVIDIA API Key: ')
embedding_model = TeradataAI(
    api_type="nim",
    model_name=getpass.getpass('Enter Embedding Model Name: '),
    api_base=getpass.getpass('Enter Embedding Model API Base URL: ')
)

print(f"✓ Embedding Model: {embedding_model.model_name}")
print("✓ Embeddings will be generated automatically during ingestion")

✓ Embedding Model: nvidia/llama-3.2-nv-embedqa-1b-v2
✓ Embeddings will be generated automatically during ingestion


## <b style='font-size:20px;font-family:Arial'>4. Prepare Parquet Input Data with Pre-computed Embeddings</b>

Load Parquet files that contain pre-computed embeddings. You can load from local storage, S3, or Azure Blob Storage.

## <b style='font-size:20px;font-family:Arial'>4a. Load from Local Storage</b>

Using Local Parquet files with pre-computed embeddings

In [ ]:
# Use existing combined Parquet file with embeddings
notebook_dir = os.getcwd()
data_dir = os.path.join(os.path.dirname(notebook_dir), "example-data", "parquet_files")
parquet_file_path = os.path.join(data_dir, "all_documents_with_embeddings.parquet")

# Verify file exists
if not os.path.exists(parquet_file_path):
    raise FileNotFoundError(f"Combined Parquet file not found: {parquet_file_path}")

print(f"✓ Loading Parquet file: {os.path.basename(parquet_file_path)}")
print(f"✓ File location: {data_dir}")

# Verify the schema - should include 'embedding' column
parquet_file = pq.ParquetFile(parquet_file_path)
schema = parquet_file.schema_arrow
print(f"\n✓ Schema verification:")
print(f"  Columns: {schema.names}")
print(f"  Types: {[str(field.type) for field in schema]}")
if 'embedding' in schema.names:
    print("  ✓ 'embedding' column found")
else:
    print("  ⚠ WARNING: 'embedding' column not found!")

# Configure local file source
local_config = LocalConfig(
    files=[parquet_file_path],
    files_type="parquet"
)
print(f"\n✓ LocalConfig created for local file ingestion")

✓ Loading Parquet file: all_documents_with_embeddings.parquet
✓ File location: ./sample_data/test_data_ingestor_parquet_embedding

✓ Schema verification:
  Columns: ['id', 'doc_title', 'content', 'category', 'author', 'publish_date', 'embedding']
  Types: ['string', 'string', 'string', 'string', 'string', 'string', 'list<element: float>']
  ✓ 'embedding' column found

✓ LocalConfig created for local file ingestion


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_local = f"parquet_file_embedding_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema (includes embedding column for pre-computed embeddings)
extraction_schema_local = ExtractionSchema(
    table_name=f"parquet_embedding_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="content", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200)),
        ColumnInfo(name="publish_date", datatype=VARCHAR(500))
    ],
    embedding_columns=ColumnInfo(name="embedding")  # Changed to singular - matches Parquet file schema
)

# Build Ingestor pipeline (note: uses .embed() instead of .upsert())
ingestor_local = (
    Ingestor(
        name=collection_name_local,
        type=CollectionType.FILE_EMBEDDING_BASED,
        target_database="vsdemo01",
        description="File-Embedding-Based collection from Parquet files with pre-computed embeddings",
        embedding_model=embedding_model
    )
    .files(
        files=local_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_local
    )
    .upsert()
)

print(f"✓ Collection Name: {collection_name_local}")
print("✓ Collection Type: FILE-EMBEDDING-BASED")
print(f"✓ Storage: {type(local_config).__name__}")
print("✓ Ingestor pipeline configured (using pre-computed embeddings)")

✓ Collection Name: parquet_file_embedding_20260217_173031
✓ Collection Type: FILE-EMBEDDING-BASED
✓ Storage: LocalConfig
✓ Ingestor pipeline configured (using pre-computed embeddings)


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [6]:
# Check if collection already exists and destroy it
try:
    existing_collection_local = Collection(name=collection_name_local)
    if existing_collection_local.exists:
        print(f"⚠ Collection '{collection_name_local}' already exists. Destroying it first...")
        existing_collection_local.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)  # Wait for cleanup
except Exception as e:
    pass

# Execute ingestion (creates collection and ingests embeddings)
print("Starting ingestion pipeline...")
result_local = ingestor_local.run()
print("✓ Ingestion completed - embeddings uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with default settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/1

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_local = existing_collection_local.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

# Access the actual results
results_list = search_results_local.similar_objects
result_count = search_results_local.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Content: {result.get('content', '')[:150]}...")


Search Results (3 documents):

1. Cloud Storage Best Practices
   Author: Alice Johnson
   Category: cloud
   Content: Modern cloud storage solutions like Amazon S3, Azure Blob Storage, and Google Cloud Storage offer scalable, secure, and cost-effective data storage. B...

2. Retrieval Augmented Generation (RAG)
   Author: Bob Wilson
   Category: ai
   Content: RAG combines large language models with external knowledge retrieval to generate more accurate and contextually relevant responses. The system retriev...

3. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Content: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_local.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_local}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_parquet_file_embedding_20260217_173031_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,parquet_embedding_table_20260217173031,2,all_documents_with_embeddings.parquet,"0.0262893959879875,-0.0110127991065383,-0.0283601228147745,-0.0214371811598539,0.00263899867422879,0.0900866612792015,-0.0063478765077889,0.028549337759614,-0.0210445877164602,-0.0787114650011063,-0.0689935013651848,0.0116009246557951,-0.0379553623497486,0.0804309919476509,-0.00447172205895185,-0.0614839680492878,-0.101325735449791,0.0311820954084396,-0.0116831371560693,-0.0210227593779564,0.0826897546648979,-0.0650421679019928,0.112006090581417,0.079835832118988,0.0535124018788338,0.0452006831765175,0.0250275656580925,0.0544417351484299,-0.0108652105554938,0.0023815855383873,0.015121815726161,0.0504669062793255,0.0286510456353426,0.0153274042531848,0.0217059850692749,0.0933705121278763,-0.00601900275796652,0.00782338250428438,0.0470376312732697,0.131491243839264,0.0224598813802004,0.0148789566010237,-0.0229822900146246,-0.101460419595242,0.0553334131836891,0.0479838065803051,0.00620240345597267,0.0252716541290283,-0.0165113992989063,-0.0255850031971931,-0.12274032086134,0.021840950474143,-0.100737884640694,0","0.0262893959879875,-0.0110127991065383,-0.0283601228147745,-0.0214371811598539,0.00263899867422879,0.0900866612792015,-0.0063478765077889,0.028549337759614,-0.0210445877164602,-0.0787114650011063,-0.0689935013651848,0.0116009246557951,-0.0379553623497486,0.0804309919476509,-0.00447172205895185,-0.0614839680492878,-0.101325735449791,0.0311820954084396,-0.0116831371560693,-0.0210227593779564,0.0826897546648979,-0.0650421679019928,0.112006090581417,0.079835832118988,0.0535124018788338,0.0452006831765175,0.0250275656580925,0.0544417351484299,-0.0108652105554938,0.0023815855383873,0.015121815726161,0.0504669062793255,0.0286510456353426,0.0153274042531848,0.0217059850692749,0.0933705121278763,-0.00601900275796652,0.00782338250428438,0.0470376312732697,0.131491243839264,0.0224598813802004,0.0148789566010237,-0.0229822900146246,-0.101460419595242,0.0553334131836891,0.0479838065803051,0.00620240345597267,0.0252716541290283,-0.0165113992989063,-0.0255850031971931,-0.12274032086134,0.021840950474143,-0.100737884640694,0"
vsdemo01,parquet_embedding_table_20260217173031,3,all_documents_with_embeddings.parquet,"0.0287384297698736,0.0430868677794933,0.0314615033566952,-0.00734057649970055,-0.0662872344255447,0.0287791695445776,-0.0601086653769016,0.0158023238182068,0.052299503237009,-0.0301604773849249,-0.0421138294041157,-0.0471976958215237,-0.0357861220836639,0.0735822841525078,0.0201996732503176,-0.0390963852405548,-0.0753288641571999,-0.018471235409379,0.0240505672991276,0.0010572241153568,-0.0119655914604664,0.0463405884802341,-0.00213939649984241,-0.0390174426138401,0.00780295254662633,0.0626192092895508,-0.0577947124838829,-0.00296074617654085,0.0505435429513454,-0.0388379618525505,0.0706248059868813,-0.0243311244994402,0.0319158621132374,0.0655975416302681,0.0313311740756035,-0.0553291030228138,0.0838411226868629,0.0466581508517265,0.00727192359045148,0.0109874131157994,-0.0402432605624199,0.00750750210136175,-0.0209342297166586,-0.0546264536678791,-0.0982257872819901,-0.0985220596194267,-0.033162783831358,-0.0470633991062641,0.0870711132884026,0.0570798255503178,-0.0419445335865021,0.024388313293457,-0.01479","0.0287384297698736,0.0430868677794933,0.0314615033566952,-0.00734057649970055,-0.0662872344255447,0.0287791695445776,-0.0601086653769016,0.0158023238182068,0.052299503237009,-0.0301604773849249,-0.0421138294041157,-0.0471976958215237,-0.0357861220836639,0.0735822841525078,0.0201996732503176,-0.0390963852405548,-0.0753288641571999,-0.018471235409379,0.0240505672991276,0.0010572241153568,-0.0119655914604664,0.0463405884802341,-0.00213939649984241,-0.0390174426138401,0.00780295254662633,0.0626192092895508,-0.0577947124838829,-0.00296074617654085,0.0505435429513454,-0.0388379618525505,0.0706248059868813,-0.0243311244994402,0.0319158621132374,0.0655975416302681

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_local.destroy()
print(f"✓ Collection '{collection_name_local}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4b. Load from S3 Storage</b>

Instead of local files, you can ingest Parquet files with pre-computed embeddings directly from Amazon S3.

In [ ]:
from teradatagenai.vector_store import S3Config

# S3 Configuration
s3_config = S3Config(
    bucket=getpass.getpass('Enter S3 Bucket: '),
    key=getpass.getpass('Enter S3 Key: '),
    aws_access_key_id=getpass.getpass('Enter AWS Access Key ID: '),
    aws_secret_access_key=getpass.getpass('Enter AWS Secret Access Key: '),
    region_name=getpass.getpass('Enter AWS Region: '),
    files_type="parquet",
    aws_session_token=getpass.getpass('Enter AWS Session Token: ')
)

print("✓ S3 storage configuration enabled")
print(f"  Bucket: {s3_config.bucket}")
print(f"  Key: {s3_config.key}")
print(f"  Region: {s3_config.region_name}")

✓ S3 storage configuration enabled
  Bucket: genaitestaanchal
  Key: files/all_documents_with_embeddings.parquet
  Region: us-west-2


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_s3 = f"parquet_file_embedding_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_s3 = ExtractionSchema(
    table_name=f"parquet_embedding_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="content", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200)),
        ColumnInfo(name="publish_date", datatype=VARCHAR(500))
    ],
    embedding_columns=ColumnInfo(name="embedding")
)

# Build Ingestor pipeline
ingestor_s3 = (
    Ingestor(
        name=collection_name_s3,
        type=CollectionType.FILE_EMBEDDING_BASED,
        target_database="vsdemo01",
        description="File-Embedding-Based collection from Parquet files with pre-computed embeddings",
        embedding_model=embedding_model
    )
    .files(
        files=s3_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_s3
    )
    .upsert()
)

print(f"✓ Collection Name: {collection_name_s3}")
print("✓ Collection Type: FILE-EMBEDDING-BASED")
print(f"✓ Storage: {type(s3_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: parquet_file_embedding_20260217_173558
✓ Collection Type: FILE-EMBEDDING-BASED
✓ Storage: S3Config
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [17]:
# Check if collection already exists and destroy it
try:
    existing_collection_s3 = Collection(name=collection_name_s3)
    if existing_collection_s3.exists:
        print(f"⚠ Collection '{collection_name_s3}' already exists. Destroying it first...")
        existing_collection_s3.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)
except Exception as e:
    pass

# Execute ingestion
print("Starting ingestion pipeline...")
result_s3 = ingestor_s3.run()
print("✓ Ingestion completed - embeddings uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'parquet_file_embedding_20260217_173558' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with default settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_s3 = existing_collection_s3.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

results_list = search_results_s3.similar_objects
result_count = search_results_s3.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Content: {result.get('content', '')[:150]}...")


Search Results (3 documents):

1. Cloud Storage Best Practices
   Author: Alice Johnson
   Category: cloud
   Content: Modern cloud storage solutions like Amazon S3, Azure Blob Storage, and Google Cloud Storage offer scalable, secure, and cost-effective data storage. B...

2. Retrieval Augmented Generation (RAG)
   Author: Bob Wilson
   Category: ai
   Content: RAG combines large language models with external knowledge retrieval to generate more accurate and contextually relevant responses. The system retriev...

3. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Content: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_s3.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_s3}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_parquet_file_embedding_20260217_173558_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,parquet_embedding_table_20260217173558,3,all_documents_with_embeddings.parquet,"0.0287384297698736,0.0430868677794933,0.0314615033566952,-0.00734057649970055,-0.0662872344255447,0.0287791695445776,-0.0601086653769016,0.0158023238182068,0.052299503237009,-0.0301604773849249,-0.0421138294041157,-0.0471976958215237,-0.0357861220836639,0.0735822841525078,0.0201996732503176,-0.0390963852405548,-0.0753288641571999,-0.018471235409379,0.0240505672991276,0.0010572241153568,-0.0119655914604664,0.0463405884802341,-0.00213939649984241,-0.0390174426138401,0.00780295254662633,0.0626192092895508,-0.0577947124838829,-0.00296074617654085,0.0505435429513454,-0.0388379618525505,0.0706248059868813,-0.0243311244994402,0.0319158621132374,0.0655975416302681,0.0313311740756035,-0.0553291030228138,0.0838411226868629,0.0466581508517265,0.00727192359045148,0.0109874131157994,-0.0402432605624199,0.00750750210136175,-0.0209342297166586,-0.0546264536678791,-0.0982257872819901,-0.0985220596194267,-0.033162783831358,-0.0470633991062641,0.0870711132884026,0.0570798255503178,-0.0419445335865021,0.024388313293457,-0.01479","0.0287384297698736,0.0430868677794933,0.0314615033566952,-0.00734057649970055,-0.0662872344255447,0.0287791695445776,-0.0601086653769016,0.0158023238182068,0.052299503237009,-0.0301604773849249,-0.0421138294041157,-0.0471976958215237,-0.0357861220836639,0.0735822841525078,0.0201996732503176,-0.0390963852405548,-0.0753288641571999,-0.018471235409379,0.0240505672991276,0.0010572241153568,-0.0119655914604664,0.0463405884802341,-0.00213939649984241,-0.0390174426138401,0.00780295254662633,0.0626192092895508,-0.0577947124838829,-0.00296074617654085,0.0505435429513454,-0.0388379618525505,0.0706248059868813,-0.0243311244994402,0.0319158621132374,0.0655975416302681,0.0313311740756035,-0.0553291030228138,0.0838411226868629,0.0466581508517265,0.00727192359045148,0.0109874131157994,-0.0402432605624199,0.00750750210136175,-0.0209342297166586,-0.0546264536678791,-0.0982257872819901,-0.0985220596194267,-0.033162783831358,-0.0470633991062641,0.0870711132884026,0.0570798255503178,-0.0419445335865021,0.024388313293457,-0.01479"
vsdemo01,parquet_embedding_table_20260217173558,2,all_documents_with_embeddings.parquet,"0.0262893959879875,-0.0110127991065383,-0.0283601228147745,-0.0214371811598539,0.00263899867422879,0.0900866612792015,-0.0063478765077889,0.028549337759614,-0.0210445877164602,-0.0787114650011063,-0.0689935013651848,0.0116009246557951,-0.0379553623497486,0.0804309919476509,-0.00447172205895185,-0.0614839680492878,-0.101325735449791,0.0311820954084396,-0.0116831371560693,-0.0210227593779564,0.0826897546648979,-0.0650421679019928,0.112006090581417,0.079835832118988,0.0535124018788338,0.0452006831765175,0.0250275656580925,0.0544417351484299,-0.0108652105554938,0.0023815855383873,0.015121815726161,0.0504669062793255,0.0286510456353426,0.0153274042531848,0.0217059850692749,0.0933705121278763,-0.00601900275796652,0.00782338250428438,0.0470376312732697,0.131491243839264,0.0224598813802004,0.0148789566010237,-0.0229822900146246,-0.101460419595242,0.0553334131836891,0.0479838065803051,0.00620240345597267,0.0252716541290283,-0.0165113992989063,-0.0255850031971931,-0.12274032086134,0.021840950474143,-0.100737884640694,0","0.0262893959879875,-0.0110127991065383,-0.0283601228147745,-0.0214371811598539,0.00263899867422879,0.0900866612792015,-0.0063478765077889,0.028549337759614,-0.0210445877164602,-0.0787114650011063,-0.0689935013651848,0.0116009246557951,-0.0379553623497486,0.0804309919476509,-0.00447172205895185,-0.0614839680492878,-0.101325735449791,0.0311820954084396,-0.0116831371560693,-0.0210227593779564,0.0826897546648979,-0.0650421679019928,0.112006090581417,0.079835832118988,0.0535124018788338,0.0452006831765175,0.0250275656580925,0.0544417351484299,-0.0108652105554938,0.0023815855383873,0.015121815726161,0.0504669062793255,0.0286510456353426,0.0153274042531848,0.0217

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_s3.destroy()
print(f"✓ Collection '{collection_name_s3}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4c. Load from Azure Blob Storage</b>

Instead of local files or S3, you can ingest Parquet files with pre-computed embeddings from Azure Blob Storage.

In [ ]:
from teradatagenai.vector_store import AzureBlobConfig

# Azure Blob Configuration
azure_config = AzureBlobConfig(
    container=getpass.getpass('Enter Azure Container: '),
    blob_name=getpass.getpass('Enter Blob Name: '),
    account_name=getpass.getpass('Enter Azure Account Name: '),
    account_key=getpass.getpass('Enter Azure Account Key: '),
    files_type="parquet"
)

print("✓ Azure Blob storage configuration provided")
print(f"  Container: {azure_config.container}")
print(f"  Blob: {azure_config.blob_name}")
print(f"  Account: {azure_config.account_name}")

✓ Azure Blob storage configuration provided
  Container: genaitestcontainer
  Blob: all_documents_with_embeddings.parquet
  Account: genaitestaanchal


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_blob = f"parquet_file_embedding_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_blob = ExtractionSchema(
    table_name=f"parquet_embedding_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="content", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200)),
        ColumnInfo(name="publish_date", datatype=VARCHAR(500))
    ],
    embedding_columns=ColumnInfo(name="embedding")
)

# Build Ingestor pipeline
ingestor_blob = (
    Ingestor(
        name=collection_name_blob,
        type=CollectionType.FILE_EMBEDDING_BASED,
        target_database="vsdemo01",
        description="File-Embedding-Based collection from Parquet files with pre-computed embeddings",
        embedding_model=embedding_model
    )
    .files(
        files=azure_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_blob
    )
    .upsert()
)

print(f"✓ Collection Name: {collection_name_blob}")
print("✓ Collection Type: FILE-EMBEDDING-BASED")
print(f"✓ Storage: {type(azure_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: parquet_file_embedding_20260217_174002
✓ Collection Type: FILE-EMBEDDING-BASED
✓ Storage: AzureBlobConfig
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [22]:
# Check if collection already exists and destroy it
try:
    existing_collection_blob = Collection(name=collection_name_blob)
    if existing_collection_blob.exists:
        print(f"⚠ Collection '{collection_name_blob}' already exists. Destroying it first...")
        existing_collection_blob.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)
except Exception as e:
    pass

# Execute ingestion
print("Starting ingestion pipeline...")
result_blob = ingestor_blob.run()
print("✓ Ingestion completed - embeddings uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'parquet_file_embedding_20260217_174002' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with default settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_blob = existing_collection_blob.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

results_list = search_results_blob.similar_objects
result_count = search_results_blob.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Content: {result.get('content', '')[:150]}...")


Search Results (3 documents):

1. Cloud Storage Best Practices
   Author: Alice Johnson
   Category: cloud
   Content: Modern cloud storage solutions like Amazon S3, Azure Blob Storage, and Google Cloud Storage offer scalable, secure, and cost-effective data storage. B...

2. Retrieval Augmented Generation (RAG)
   Author: Bob Wilson
   Category: ai
   Content: RAG combines large language models with external knowledge retrieval to generate more accurate and contextually relevant responses. The system retriev...

3. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Content: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_blob.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_blob}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_parquet_file_embedding_20260217_174002_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,parquet_embedding_table_20260217174002,2,all_documents_with_embeddings.parquet,"0.0262893959879875,-0.0110127991065383,-0.0283601228147745,-0.0214371811598539,0.00263899867422879,0.0900866612792015,-0.0063478765077889,0.028549337759614,-0.0210445877164602,-0.0787114650011063,-0.0689935013651848,0.0116009246557951,-0.0379553623497486,0.0804309919476509,-0.00447172205895185,-0.0614839680492878,-0.101325735449791,0.0311820954084396,-0.0116831371560693,-0.0210227593779564,0.0826897546648979,-0.0650421679019928,0.112006090581417,0.079835832118988,0.0535124018788338,0.0452006831765175,0.0250275656580925,0.0544417351484299,-0.0108652105554938,0.0023815855383873,0.015121815726161,0.0504669062793255,0.0286510456353426,0.0153274042531848,0.0217059850692749,0.0933705121278763,-0.00601900275796652,0.00782338250428438,0.0470376312732697,0.131491243839264,0.0224598813802004,0.0148789566010237,-0.0229822900146246,-0.101460419595242,0.0553334131836891,0.0479838065803051,0.00620240345597267,0.0252716541290283,-0.0165113992989063,-0.0255850031971931,-0.12274032086134,0.021840950474143,-0.100737884640694,0","0.0262893959879875,-0.0110127991065383,-0.0283601228147745,-0.0214371811598539,0.00263899867422879,0.0900866612792015,-0.0063478765077889,0.028549337759614,-0.0210445877164602,-0.0787114650011063,-0.0689935013651848,0.0116009246557951,-0.0379553623497486,0.0804309919476509,-0.00447172205895185,-0.0614839680492878,-0.101325735449791,0.0311820954084396,-0.0116831371560693,-0.0210227593779564,0.0826897546648979,-0.0650421679019928,0.112006090581417,0.079835832118988,0.0535124018788338,0.0452006831765175,0.0250275656580925,0.0544417351484299,-0.0108652105554938,0.0023815855383873,0.015121815726161,0.0504669062793255,0.0286510456353426,0.0153274042531848,0.0217059850692749,0.0933705121278763,-0.00601900275796652,0.00782338250428438,0.0470376312732697,0.131491243839264,0.0224598813802004,0.0148789566010237,-0.0229822900146246,-0.101460419595242,0.0553334131836891,0.0479838065803051,0.00620240345597267,0.0252716541290283,-0.0165113992989063,-0.0255850031971931,-0.12274032086134,0.021840950474143,-0.100737884640694,0"
vsdemo01,parquet_embedding_table_20260217174002,3,all_documents_with_embeddings.parquet,"0.0287384297698736,0.0430868677794933,0.0314615033566952,-0.00734057649970055,-0.0662872344255447,0.0287791695445776,-0.0601086653769016,0.0158023238182068,0.052299503237009,-0.0301604773849249,-0.0421138294041157,-0.0471976958215237,-0.0357861220836639,0.0735822841525078,0.0201996732503176,-0.0390963852405548,-0.0753288641571999,-0.018471235409379,0.0240505672991276,0.0010572241153568,-0.0119655914604664,0.0463405884802341,-0.00213939649984241,-0.0390174426138401,0.00780295254662633,0.0626192092895508,-0.0577947124838829,-0.00296074617654085,0.0505435429513454,-0.0388379618525505,0.0706248059868813,-0.0243311244994402,0.0319158621132374,0.0655975416302681,0.0313311740756035,-0.0553291030228138,0.0838411226868629,0.0466581508517265,0.00727192359045148,0.0109874131157994,-0.0402432605624199,0.00750750210136175,-0.0209342297166586,-0.0546264536678791,-0.0982257872819901,-0.0985220596194267,-0.033162783831358,-0.0470633991062641,0.0870711132884026,0.0570798255503178,-0.0419445335865021,0.024388313293457,-0.01479","0.0287384297698736,0.0430868677794933,0.0314615033566952,-0.00734057649970055,-0.0662872344255447,0.0287791695445776,-0.0601086653769016,0.0158023238182068,0.052299503237009,-0.0301604773849249,-0.0421138294041157,-0.0471976958215237,-0.0357861220836639,0.0735822841525078,0.0201996732503176,-0.0390963852405548,-0.0753288641571999,-0.018471235409379,0.0240505672991276,0.0010572241153568,-0.0119655914604664,0.0463405884802341,-0.00213939649984241,-0.0390174426138401,0.00780295254662633,0.0626192092895508,-0.0577947124838829,-0.00296074617654085,0.0505435429513454,-0.0388379618525505,0.0706248059868813,-0.0243311244994402,0.0319158621132374,0.0655975416302681

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_blob.destroy()
print(f"✓ Collection '{collection_name_blob}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4d. Load from Google Cloud Storage</b>

Instead of local files, S3, or Azure, you can ingest Parquet files with pre-computed embeddings from Google Cloud Platform (GCP) Storage.

In [ ]:
from teradatagenai import GCPConfig

# GCP Configuration
gcp_config = GCPConfig(
    files_type="parquet",
    bucket=getpass.getpass('Enter GCP Bucket: '),
    blob_name=getpass.getpass('Enter GCP Blob Name: '),
    project_id=getpass.getpass('Enter GCP Project ID: '),
    secret=json.loads(getpass.getpass('Enter GCP Service Account JSON (paste entire JSON): '))
)

print("✓ GCP storage configuration enabled")
print(f"  Bucket: {gcp_config.bucket}")
print(f"  Blob: {gcp_config.blob_name}")
print(f"  Project: {gcp_config.project_id}")

✓ GCP storage configuration enabled
  Bucket: ak250090-filestore
  Blob: harsha_files/all_documents_with_embeddings.parquet
  Project: icgcp-vector-store


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_gcp = f"parquet_file_embedding_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema (includes embedding column for pre-computed embeddings)
extraction_schema_gcp = ExtractionSchema(
    table_name=f"parquet_embedding_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="content", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200)),
        ColumnInfo(name="publish_date", datatype=VARCHAR(500))
    ],
    embedding_columns=ColumnInfo(name="embedding")
)

# Build Ingestor pipeline (note: uses .embed() instead of .upsert())
ingestor_gcp = (
    Ingestor(
        name=collection_name_gcp,
        type=CollectionType.FILE_EMBEDDING_BASED,
        target_database="vsdemo01",
        description="File-Embedding-Based collection from Parquet files with pre-computed embeddings",
        embedding_model=embedding_model
    )
    .files(
        files=gcp_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_gcp
    )
    .upsert()
)

print(f"✓ Collection Name: {collection_name_gcp}")
print("✓ Collection Type: FILE-EMBEDDING-BASED")
print(f"✓ Storage: {type(gcp_config).__name__}")
print("✓ Ingestor pipeline configured (using pre-computed embeddings)")

✓ Collection Name: parquet_file_embedding_20260217_174236
✓ Collection Type: FILE-EMBEDDING-BASED
✓ Storage: GCPConfig
✓ Ingestor pipeline configured (using pre-computed embeddings)


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [27]:
# Check if collection already exists and destroy it
try:
    existing_collection_gcp = Collection(name=collection_name_gcp)
    if existing_collection_gcp.exists:
        print(f"⚠ Collection '{collection_name_gcp}' already exists. Destroying it first...")
        existing_collection_gcp.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)
except Exception as e:
    pass

# Execute ingestion
print("Starting ingestion pipeline...")
result_gcp = ingestor_gcp.run()
print("✓ Ingestion completed - embeddings uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'parquet_file_embedding_20260217_174236' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with default settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_gcp = existing_collection_gcp.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

results_list = search_results_gcp.similar_objects
result_count = search_results_gcp.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Content: {result.get('content', '')[:150]}...")


Search Results (3 documents):

1. Cloud Storage Best Practices
   Author: Alice Johnson
   Category: cloud
   Content: Modern cloud storage solutions like Amazon S3, Azure Blob Storage, and Google Cloud Storage offer scalable, secure, and cost-effective data storage. B...

2. Retrieval Augmented Generation (RAG)
   Author: Bob Wilson
   Category: ai
   Content: RAG combines large language models with external knowledge retrieval to generate more accurate and contextually relevant responses. The system retriev...

3. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Content: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_gcp.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_gcp}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_parquet_file_embedding_20260217_174236_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,parquet_embedding_table_20260217174236,2,all_documents_with_embeddings.parquet,"0.0262893959879875,-0.0110127991065383,-0.0283601228147745,-0.0214371811598539,0.00263899867422879,0.0900866612792015,-0.0063478765077889,0.028549337759614,-0.0210445877164602,-0.0787114650011063,-0.0689935013651848,0.0116009246557951,-0.0379553623497486,0.0804309919476509,-0.00447172205895185,-0.0614839680492878,-0.101325735449791,0.0311820954084396,-0.0116831371560693,-0.0210227593779564,0.0826897546648979,-0.0650421679019928,0.112006090581417,0.079835832118988,0.0535124018788338,0.0452006831765175,0.0250275656580925,0.0544417351484299,-0.0108652105554938,0.0023815855383873,0.015121815726161,0.0504669062793255,0.0286510456353426,0.0153274042531848,0.0217059850692749,0.0933705121278763,-0.00601900275796652,0.00782338250428438,0.0470376312732697,0.131491243839264,0.0224598813802004,0.0148789566010237,-0.0229822900146246,-0.101460419595242,0.0553334131836891,0.0479838065803051,0.00620240345597267,0.0252716541290283,-0.0165113992989063,-0.0255850031971931,-0.12274032086134,0.021840950474143,-0.100737884640694,0","0.0262893959879875,-0.0110127991065383,-0.0283601228147745,-0.0214371811598539,0.00263899867422879,0.0900866612792015,-0.0063478765077889,0.028549337759614,-0.0210445877164602,-0.0787114650011063,-0.0689935013651848,0.0116009246557951,-0.0379553623497486,0.0804309919476509,-0.00447172205895185,-0.0614839680492878,-0.101325735449791,0.0311820954084396,-0.0116831371560693,-0.0210227593779564,0.0826897546648979,-0.0650421679019928,0.112006090581417,0.079835832118988,0.0535124018788338,0.0452006831765175,0.0250275656580925,0.0544417351484299,-0.0108652105554938,0.0023815855383873,0.015121815726161,0.0504669062793255,0.0286510456353426,0.0153274042531848,0.0217059850692749,0.0933705121278763,-0.00601900275796652,0.00782338250428438,0.0470376312732697,0.131491243839264,0.0224598813802004,0.0148789566010237,-0.0229822900146246,-0.101460419595242,0.0553334131836891,0.0479838065803051,0.00620240345597267,0.0252716541290283,-0.0165113992989063,-0.0255850031971931,-0.12274032086134,0.021840950474143,-0.100737884640694,0"
vsdemo01,parquet_embedding_table_20260217174236,3,all_documents_with_embeddings.parquet,"0.0287384297698736,0.0430868677794933,0.0314615033566952,-0.00734057649970055,-0.0662872344255447,0.0287791695445776,-0.0601086653769016,0.0158023238182068,0.052299503237009,-0.0301604773849249,-0.0421138294041157,-0.0471976958215237,-0.0357861220836639,0.0735822841525078,0.0201996732503176,-0.0390963852405548,-0.0753288641571999,-0.018471235409379,0.0240505672991276,0.0010572241153568,-0.0119655914604664,0.0463405884802341,-0.00213939649984241,-0.0390174426138401,0.00780295254662633,0.0626192092895508,-0.0577947124838829,-0.00296074617654085,0.0505435429513454,-0.0388379618525505,0.0706248059868813,-0.0243311244994402,0.0319158621132374,0.0655975416302681,0.0313311740756035,-0.0553291030228138,0.0838411226868629,0.0466581508517265,0.00727192359045148,0.0109874131157994,-0.0402432605624199,0.00750750210136175,-0.0209342297166586,-0.0546264536678791,-0.0982257872819901,-0.0985220596194267,-0.033162783831358,-0.0470633991062641,0.0870711132884026,0.0570798255503178,-0.0419445335865021,0.024388313293457,-0.01479","0.0287384297698736,0.0430868677794933,0.0314615033566952,-0.00734057649970055,-0.0662872344255447,0.0287791695445776,-0.0601086653769016,0.0158023238182068,0.052299503237009,-0.0301604773849249,-0.0421138294041157,-0.0471976958215237,-0.0357861220836639,0.0735822841525078,0.0201996732503176,-0.0390963852405548,-0.0753288641571999,-0.018471235409379,0.0240505672991276,0.0010572241153568,-0.0119655914604664,0.0463405884802341,-0.00213939649984241,-0.0390174426138401,0.00780295254662633,0.0626192092895508,-0.0577947124838829,-0.00296074617654085,0.0505435429513454,-0.0388379618525505,0.0706248059868813,-0.0243311244994402,0.0319158621132374,0.0655975416302681

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_gcp.destroy()
print(f"✓ Collection '{collection_name_gcp}' destroyed")

---

## Summary

This notebook demonstrated how to:
- Create a FILE-EMBEDDING-BASED vector collection from Parquet files with pre-computed embeddings
- Configure the Ingestor pipeline with `embedding_columns` parameter in `ExtractionSchema`
- Test with four different storage types: Local, S3, Azure Blob, and GCP
- Implement collection readiness polling to handle async indexing
- Perform semantic similarity search on the collection

**Key Points:**
- Parquet files must contain pre-computed embeddings in the `embedding` column (singular)
- Use `embedding_columns=ColumnInfo(name="embedding")` in `ExtractionSchema` with `.upsert()` method
- Collections require indexing time after ingestion before similarity search is available

- Metadata columns are preserved for filtering and retrieval- Each storage type (Local/S3/Azure/GCP) has its own complete isolated workflow